# Hangman — blend whatever checkpoints exist (no training)

The twin (`hangman_r3b.pt`) is already trained. This notebook only loads the
checkpoints it can find, finds the best blend on the full 10,000-word holdout,
and — if the gain is real — scores `test.txt` and writes a submission.

**~20 minutes, no training.** Unlike the last run, this one discovers what is
actually in the dataset instead of assuming, so a missing file cannot kill it.

### Before running

`hangman_r3b.pt` must be reachable. Get it from the failed run's **Output** tab
into a dataset — either add it to `hangman-weights` as a new version *alongside*
`hangman_r3.pt`, or make a separate dataset and attach both. `hangman_r2.pt` is
optional; if you still have it, add it too and the sweep will try it.

| | holdout | test.txt | public LB |
|---|---|---|---|
| round 3 | 67.28% | 70.097% | 70.0334 |
| best blend | ? | ? | ? |


In [ ]:
import sys, os, glob, json, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch

# --- find the source dataset instead of assuming its path ---
cands = glob.glob("/kaggle/input/**/hangman/encoding.py", recursive=True)
assert cands, "hangman-src dataset not attached (no hangman/encoding.py under /kaggle/input)"
SRC = os.path.dirname(os.path.dirname(cands[0]))
sys.path.insert(0, SRC)
print("SRC =", SRC)

from hangman.data import load_words, overlap, split_holdout, find_competition_dir
from hangman.model import HangmanNet, ModelConfig
from hangman.policies import NeuralPolicy, FallbackPolicy
from hangman.evaluate import evaluate, print_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

# --- find every .pt anywhere under /kaggle/input — no hard-coded dataset path ---
found = {}
for path in glob.glob("/kaggle/input/**/*.pt", recursive=True):
    found[os.path.basename(path)] = path
print("checkpoints visible:")
for k, v in sorted(found.items()):
    print(f"  {k:<20} {v}")

assert "hangman_r3.pt" in found, "hangman_r3.pt not attached"
assert "hangman_r3b.pt" in found, "hangman_r3b.pt not attached - add it from the failed run's Output tab"

In [ ]:
COMP = find_competition_dir()
train_words_all = load_words(f"{COMP}/train.txt")
test_words = load_words(f"{COMP}/test.txt")
assert overlap(train_words_all, test_words)["test_words_in_train"] == 0
MAX_LEN = max(max(map(len, train_words_all)), max(map(len, test_words)))

# seed=0: the same 10k holdout every previous run used.
_, HOLDOUT = split_holdout(train_words_all, n_holdout=10_000, seed=0)
print(len(HOLDOUT), "holdout words, MAX_LEN =", MAX_LEN)

def load(name, fusion):
    ck = torch.load(found[name], map_location=DEVICE)
    net = HangmanNet(ModelConfig(**ck["cfg"]))
    net.load_state_dict(ck["model"])
    return NeuralPolicy(net, DEVICE, fusion=fusion)

p3  = load("hangman_r3.pt", 0.6)
p3b = load("hangman_r3b.pt", 0.6)
p2  = load("hangman_r2.pt", 0.3) if "hangman_r2.pt" in found else None
print("loaded r3, r3b" + (", r2" if p2 else "  (r2 absent — fine, it was optional)"))

## Sweep on the full holdout

`SOLO_R3 = 67.28` is round 3 measured on this exact holdout. The twin should
land within about a point of it; if it is far below, its training went wrong and
the blend is not trustworthy.

In [ ]:
SOLO_R3 = 67.28

candidates = {"round 3 alone": p3, "twin alone": p3b}
for w in (0.5, 0.4, 0.6):
    candidates[f"r3+twin w={w}"] = FallbackPolicy(p3, p3b, weight=w)
if p2 is not None:
    pair = FallbackPolicy(p3, p3b, weight=0.5)
    for w in (0.8, 0.7):
        candidates[f"(r3+twin)+r2 w={w}"] = FallbackPolicy(pair, p2, weight=w)

scored = []
for name, pol in candidates.items():
    t = time.time()
    m = evaluate(HOLDOUT, pol, max_len=MAX_LEN)
    scored.append((m["win_rate"], -m["mean_wrong"], name, pol))
    print(f"{name:<22} win {m['win_rate']:.2f}%  strikes {m['mean_wrong']:.3f}"
          f"  [{time.time()-t:.0f}s]")

best_win, _, best_name, best_pol = max(scored, key=lambda r: (r[0], r[1]))
gain = best_win - SOLO_R3
print(f"\nbest: {best_name} -> {best_win:.2f}%   round 3 alone {SOLO_R3}%   gain {gain:+.2f}")
print("PROCEED" if gain >= 0.3 else "STOP - inside the noise, keep the 70.0334 submission")

## Score on test.txt and write the submission

Runs only if the gain cleared 0.3. Submit only if the printed number beats
**70.097**, and remember to pick `submission.csv` explicitly in the submit dialog.

In [ ]:
assert gain >= 0.3, "holdout gain is inside the noise; keep round 3"

from hangman.submit import write_submission, validate_submission

metrics = evaluate(test_words, best_pol, max_len=MAX_LEN)
print_report(metrics)
print(f"\nround 3 alone was 70.097% / 3.433 strikes")
print(f"{best_name} is {metrics['win_rate']:.3f}% / {metrics['mean_wrong']:.3f} strikes")

write_submission(metrics["guesses"], "submission.csv")
validate_submission("submission.csv", expected_rows=len(test_words))
json.dump({"best": best_name, "holdout": best_win, "test": metrics["win_rate"]},
          open("/kaggle/working/blend_result.json", "w"))